# Armar el índice RAG

Cuaderno 1 de 2. Lee los PDF de la carpeta compartida, los trocea (1000/150), genera embeddings HuggingFace y guarda el índice Chroma en disco local; al terminar lo publica en la carpeta compartida. Reanudable por manifiesto: si se interrumpe, la siguiente corrida sigue donde quedó. Basado en la [actividad base](https://github.com/JheisonMB/ml/blob/main/langgraph-rag-referencias-astro/langgraph_rag_agente_referencias.ipynb).

In [11]:
%pip install -q langchain langchain-chroma langchain-huggingface langchain-text-splitters sentence-transformers pypdf pandas gdown

## 1. Carpeta compartida

Monta Drive y ubica `rag_corpus` (o descarga la copia pública). No pide clave de modelo: indexar no la necesita.

In [12]:
import os, shutil, subprocess

DEST = '/content/rag_corpus'
FOLDER_URL = 'https://drive.google.com/drive/folders/1dc94vK-VU1SOLpiQ8aHteH7Vah21n3_s'

from google.colab import drive
drive.mount('/content/drive')

candidates = ['/content/drive/MyDrive/rag_corpus']
found = subprocess.run(['find', '/content/drive/MyDrive', '-maxdepth', '3',
                        '-type', 'd', '-name', 'rag_corpus'],
                       capture_output=True, text=True).stdout.split()
candidates += [c for c in found if c not in candidates]
BASE = next((c for c in candidates if os.path.exists(f'{c}/pdfs')), None)
if BASE is None:
    if not os.path.exists(f'{DEST}/pdfs'):
        print('Descargando carpeta compartida...')
        import gdown
        gdown.download_folder(FOLDER_URL, output=DEST, quiet=False)
    else:
        print('Carpeta ya descargada en', DEST)
    BASE = DEST
    DRIVE_INDEX = None
else:
    print('Usando carpeta en Drive:', BASE)
    DRIVE_INDEX = f'{BASE}/index'

os.environ['RAG_PDFS'] = f'{BASE}/pdfs'
os.environ['RAG_INDEX'] = '/content/rag_index'

print('PDFs:', os.environ['RAG_PDFS'])
print('Indice local:', os.environ['RAG_INDEX'])
print('Destino en Drive:', DRIVE_INDEX)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Usando carpeta en Drive: /content/drive/MyDrive/rag_corpus
PDFs: /content/drive/MyDrive/rag_corpus/pdfs
Indice local: /content/rag_index
Destino en Drive: /content/drive/MyDrive/rag_corpus/index


## 2. Pipeline de indexación

Mismo código que `rag_pipeline_drive.py` del repo.

In [13]:
"""Indexing pipeline for the deployable RAG.

Same chunking contract as the base activity
(`langgraph-rag-referencias-astro/rag_pipeline.py`):
pypdf + RecursiveCharacterTextSplitter (1000/150) + Chroma cosine.

Two deliberate changes for Colab:
1. PDFs live in a shared Drive folder (public, editor link), not in a
   local checkout. RAG_PDFS/RAG_INDEX point there; the index is
   created in that same folder so every later run finds it ready.
2. Embeddings run on HuggingFace sentence-transformers, so no Ollama
   server is needed on the Colab runtime.

build_index() is resumable: files already in the manifest are
skipped, so running the notebook twice never re-indexes.
"""

import json
import os
import re
import shutil
import sys
import time
from pathlib import Path

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

def _default_index_dir():
    env = os.environ.get("RAG_INDEX")
    if env:
        return Path(env)
    if Path("/content").is_dir():
        return Path("/content/rag_index")
    return Path(__file__).parent / "index"


PDF_DIR = Path(os.environ.get("RAG_PDFS", "pdfs"))
INDEX_DIR = _default_index_dir()
MANIFEST_PATH = INDEX_DIR / "manifest.json"
COLLECTION = "rag_core_docs_cosine"
EMBED_MODEL = os.environ.get("RAG_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
MIN_PAGE_CHARS = 200
BATCH_SIZE = 128

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def paper_metadata(pdf_path):
    year_match = re.match(r"(\d{4}|XXXX)_", pdf_path.name)
    title = re.sub(r"^(\d{4}|XXXX)_", "", pdf_path.stem).replace("_", " ")
    return {
        "source": pdf_path.name,
        "year": year_match.group(1) if year_match else "XXXX",
        "title_guess": title,
    }


def read_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for number, page in enumerate(reader.pages, start=1):
        text = re.sub(r"[ \t]+", " ", page.extract_text() or "").strip()
        if len(text) >= MIN_PAGE_CHARS:
            pages.append(Document(page_content=text, metadata={"page": number}))
    return pages


def chunk_pdf(pdf_path):
    base = paper_metadata(pdf_path)
    chunks = []
    for page in read_pages(pdf_path):
        for piece in splitter.split_documents([page]):
            piece.metadata = {**base, **piece.metadata}
            chunks.append(piece)
    for position, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = f"{base['source']}#{position}"
    return chunks


def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"indexed": {}, "failed": {}}


def pending_pdfs():
    manifest = load_manifest()
    return [p for p in sorted(PDF_DIR.glob("*.pdf")) if p.name not in manifest["indexed"]]


def _entry_stats(name, value):
    if isinstance(value, dict):
        return {"file": name, "year": name[:4] if re.match(r"(19|20)\d{2}_", name) else "XXXX",
                "chunks": int(value.get("chunks", 0)), "pages": int(value.get("pages", 0))}
    return {"file": name, "year": name[:4] if re.match(r"(19|20)\d{2}_", name) else "XXXX",
            "chunks": int(value), "pages": 0}


def pull_index(drive_index):
    """Trae el indice de Drive al disco local. Devuelve 'local', 'pulled' o 'empty'."""
    drive_index = Path(drive_index)
    if (INDEX_DIR / "manifest.json").exists():
        return "local"
    if (drive_index / "manifest.json").exists():
        INDEX_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive_index, INDEX_DIR, dirs_exist_ok=True)
        return "pulled"
    return "empty"


def publish_index(drive_index):
    """Copia el indice local ya construido a la carpeta de Drive. Devuelve bool."""
    drive_index = Path(drive_index)
    if not (INDEX_DIR / "manifest.json").exists():
        return False
    drive_index.mkdir(parents=True, exist_ok=True)
    shutil.copytree(INDEX_DIR, drive_index, dirs_exist_ok=True)
    return True


def corpus_info():
    manifest = load_manifest()
    rows = [_entry_stats(name, value) for name, value in sorted(manifest["indexed"].items())]
    return {
        "files": len(rows),
        "chunks": sum(row["chunks"] for row in rows),
        "pages": sum(row["pages"] for row in rows),
        "failed": len(manifest.get("failed", {})),
        "rows": rows,
    }


def save_manifest(manifest):
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=1, ensure_ascii=False))


def get_vectorstore():
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    return Chroma(
        collection_name=COLLECTION,
        embedding_function=HuggingFaceEmbeddings(model_name=EMBED_MODEL),
        persist_directory=str(INDEX_DIR),
        collection_metadata={"hnsw:space": "cosine"},
    )


def build_index(limit=None):
    vectorstore = get_vectorstore()
    manifest = load_manifest()
    stale = [n for n in manifest['indexed'] if not (PDF_DIR / n).exists()]
    for n in stale + [n for n in manifest['failed'] if not (PDF_DIR / n).exists()]:
        manifest['indexed'].pop(n, None); manifest['failed'].pop(n, None)
    if stale:
        save_manifest(manifest)
        print(f'limpiados del manifiesto: {len(stale)} (ya no estan en disco)')
    pdfs = sorted(PDF_DIR.glob("*.pdf"))[:limit]
    if not pdfs:
        raise FileNotFoundError(f"No hay PDFs en {PDF_DIR}. Sube los PDFs a esa carpeta de Drive primero.")
    started = time.time()
    for number, pdf_path in enumerate(pdfs, start=1):
        if pdf_path.name in manifest["indexed"]:
            continue
        try:
            chunks = chunk_pdf(pdf_path)
            if not chunks:
                raise ValueError("sin texto extraible")
            for start in range(0, len(chunks), BATCH_SIZE):
                batch = chunks[start : start + BATCH_SIZE]
                vectorstore.add_documents(batch, ids=[c.metadata["chunk_id"] for c in batch])
            pages = max(c.metadata["page"] for c in chunks)
            manifest["indexed"][pdf_path.name] = {"chunks": len(chunks), "pages": pages}
            manifest["failed"].pop(pdf_path.name, None)
        except Exception as error:
            manifest["failed"][pdf_path.name] = f"{type(error).__name__}: {error}"[:200]
        save_manifest(manifest)
        print(f"[{number}/{len(pdfs)}] {pdf_path.name[:60]} ({time.time() - started:.0f}s)", flush=True)
        if number % 10 == 0 and globals().get('DRIVE_INDEX'):
            publish_index(DRIVE_INDEX)
            print('  (progreso publicado en Drive)', flush=True)
    if manifest["failed"]:
        print(f"fallidos en esta corrida: {len(manifest['failed'])}")
        for name, cause in list(manifest["failed"].items())[:5]:
            print(f"  - {name}: {cause}")
    en_disco = len(sorted(PDF_DIR.glob('*.pdf')))
    print(f'en disco: {en_disco} | indexados: {len(manifest["indexed"])} | fallidos: {len(manifest["failed"])}')
    return vectorstore, manifest


## 3. Indexar y publicar

Procesa solo lo pendiente e imprime las causas de fallo. Al terminar copia el índice a Drive.

In [14]:
pending = pending_pdfs()
print('pendientes:', len(pending))
vs, manifest = build_index()
info = corpus_info()
print('archivos:', info['files'], '| chunks:', info['chunks'], '| paginas:', info['pages'], '| fallidos:', info['failed'])
if DRIVE_INDEX is not None:
    print('indice publicado en Drive:', publish_index(DRIVE_INDEX))
else:
    print('sin carpeta de Drive: el indice queda solo en esta sesion')


pendientes: 119


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[38/188] 2022_J_machine_learning_approach_enhancing_erosita_observati (24s)
[39/188] 2022_K_dancing_under_stars_video_denoising_starlight.pdf (74s)
[40/188] 2022_L_computational_spectrometers_enabled_nanophotonics_dee (94s)
[41/188] 2022_L_deep_learning_solar_astronomy.pdf (141s)
[42/188] 2022_L_generating_astronomical_spectra_photometry_conditiona (147s)
[43/188] 2022_MJ_deep_learning_explore_ultra_large_scale.pdf (211s)
[44/188] 2022_M_don_t_pay_attention_noise_learning.pdf (224s)
[45/188] 2022_P_deep_learning_approach_detection_noise_type.pdf (234s)
[46/188] 2022_P_deep_learning_asteroid_detection_large_astronomical.p (280s)
[49/188] 2022_S_deblending_galaxies_generative_adversarial_networks.p (289s)
[51/188] 2022_U_deep_learning_galaxy_image_deconvolution.pdf (298s)
[52/188] 2022_W_improving_spatial_resolution_solar_images_improved.pd (308s)
[53/188] 2022_YJ_application_deep_learning_solar_space_weather.pdf (318s)
[54/188] 2022_Y_autophasenn_unsupervised_physics_aware_deep_learning

[61/188] 2023_CL_matched_filtering_gravitational_wave_detection_witho (429s)
[62/188] 2023_C_detecting_denoising_gravitational_wave_signals_binary (441s)
[63/188] 2023_DI_star_image_centering_deep_learning_hst.pdf (452s)
[64/188] 2023_D_artificial_intelligence_deep_learning_advancing_pet.p (460s)
[65/188] 2023_D_denoising_diffusion_probabilistic_models_predict_dens (474s)
[67/188] 2023_D_predicting_radiation_field_molecular_clouds_denoising (488s)
[69/188] 2023_F_dawes_review_10_impact_deep_learning.pdf (544s)
[70/188] 2023_F_deep_learning_radiointerferometric_imaging_gan_aided. (558s)
[72/188] 2023_HS_machine_learning_rotational_spectra_analysis_interst (579s)
[73/188] 2023_H_review_artificial_intelligence_applications_astronomi (593s)
[74/188] 2023_I_optimal_background_removal_denoising_diffusion_models (597s)
[75/188] 2023_JF_test_artificial_neural_networks_likelihood_free.pdf (606s)
[76/188] 2023_K_weighted_anisotropicisotropic_total_variation_poisson (610s)
[77/188] 2023_L_deep_le

[91/188] 2023_S_point_spread_function_deconvolution_convolutional_aut (813s)
[92/188] 2023_T_dawning_new_era_gravitational_wave_data.pdf (836s)
[93/188] 2023_T_galaxy_image_deconvolution_weak_gravitational_lensing (842s)
[96/188] 2023_W_detection_einstein_telescope_gravitational_wave_signa (849s)
[97/188] 2023_X_attention_deep_learning_model_image_desaturation.pdf (857s)
[98/188] 2023_X_estimating_stellar_parameters_lamost_low_resolution.p (868s)
[99/188] 2023_X_foreground_removal_co_intensity_mapping_deep.pdf (880s)
[102/188] 2023_YP_recovering_cosmic_microwave_background_polarization_ (891s)
[103/188] 2023_Y_finding_needle_haystack_faint_small_space.pdf (910s)
[104/188] 2023_Y_restoration_spatially_variant_blurred_images_wide.pdf (925s)
[105/188] 2023_Z_tsdsr_temporalspatial_domain_denoise_super_resolution (936s)
[106/188] 2023_Z_wavelets_deep_learning_era.pdf (944s)
[107/188] 2024_A_bayesian_deconvolution_astronomical_images_diffusion_ (951s)
[108/188] 2024_A_cleaning_cygnus_deep_fa

[130/188] 2024_J_weak_signal_extraction_enabled_deep_neural.pdf (1243s)
[131/188] 2024_KE_spectropolarimetric_inversion_four_dimensions_deep_l (1264s)
[132/188] 2024_KS_detecting_refining_hirise_image_patches_obscured.pdf (1270s)
[133/188] 2024_K_use_deep_learning_phase_recovery.pdf (1310s)
[134/188] 2024_L_relationships_between_observables_astrophysical_condi (1474s)
[135/188] 2024_MT_flare_up_your_data_diffusion_augmentation.pdf (1484s)
[136/188] 2024_M_generating_stellar_spectra_neural_networks.pdf (1490s)
[137/188] 2024_M_model_suppressing_stray_light_astronomical_images.pdf (1501s)
[138/188] 2024_M_radio_astronomical_image_reconstruction_conditional_d (1516s)
[140/188] 2024_N_machine_learning_applications_gravitational_wave_astr (1532s)
[141/188] 2024_O_detecting_streaks_smart_telescopes_images_deep.pdf (1542s)
[142/188] 2024_O_learning_remove_noise_astronomical_images_without.pdf (1566s)
[143/188] 2024_P_detecting_anomalous_images_astronomical_datasets.pdf (1575s)
[144/188] 2024_

[154/188] 2024_WB_efficient_identification_broad_absorption_line_quasa (1709s)
[155/188] 2024_W_improving_spatial_resolution_solar_images_super.pdf (1717s)
[156/188] 2024_W_stellargan_classifying_stellar_spectra_generative_adv (1738s)
[157/188] 2024_X_deep_neural_network_restore_pulsar_dynamic.pdf (1741s)
[158/188] 2024_X_ephemeris_accuracy_improvement_moons_gas_giants.pdf (1753s)
[159/188] 2024_X_restoring_dynamic_spectra_corrupted_radio_frequency.p (1820s)
[160/188] 2024_Y_research_biomedical_image_denoising_method_deep.pdf (1979s)
[161/188] 2024_Y_self_supervised_image_denoising_third_harmonic.pdf (1992s)
[162/188] 2025_A_bcddm_branch_correction_denoising_diffusion_model.pdf (2013s)
[163/188] 2025_A_denoising_radio_pulses_air_showers_machine.pdf (2018s)
[164/188] 2025_A_gamma_ray_burst_light_curve_reconstruction.pdf (2042s)
[165/188] 2025_BD_wavelength_aware_unsupervised_learning_approach_larg (2055s)
[167/188] 2025_B_surveystart_image_denoising.pdf (2064s)
[168/188] 2025_CS_detecti

[174/188] 2025_D_super_resolving_herschel_deep_learning_deconvolution. (2196s)
[175/188] 2025_E_examining_vision_foundation_models_classification_det (2211s)


[177/188] 2025_G_efficient_unsupervised_classification_model_galaxy_mo (2227s)
fallidos en esta corrida: 2
  - 2023_D_predicting_radiation_field_molecular_clouds_denoising.pdf: LimitReachedError: Reached maximum number of tokens per stream.
  - 2024_V_debiasing_diffusion_probabilistic_reconstruction_dark_matter.pdf: LimitReachedError: Limit reached while decompressing. 13069250 bytes remaining.
archivos: 190 | chunks: 17024 | paginas: 4101 | fallidos: 2
indice publicado en Drive: True


## 4. Verificación

Totales y tabla por archivo.

In [15]:
import pandas as pd
info = corpus_info()
print(f"archivos: {info['files']} | chunks: {info['chunks']} | paginas: {info['pages']} | fallidos: {info['failed']}")
display(pd.DataFrame(info['rows']))


archivos: 190 | chunks: 17024 | paginas: 4101 | fallidos: 2


,file,year,chunks,pages
0,2021_A_astrovader_astronomical_variational_dee...,2021,123,24
1,2021_A_classification_denoising_cosmic_ray_rad...,2021,29,8
2,2021_A_deepadversaries_examining_robustness_de...,2021,107,20
3,2021_A_learning_denoise_astronomical_images_u_...,2021,60,12
4,2021_B_deep_learning_approach_quasar_continuum...,2021,122,24
...,...,...,...,...
185,2025_S_moments_matter_posterior_recovery_poiss...,2025,23,7
186,2025_X_review_deep_learning_analyses_impact_cr...,2025,146,21
187,2026_Liu_denoising_the_deep_sky_physics_based_...,2026,65,19
188,2026_Y_background_intensity_estimation_cassini...,2026,47,7
